In [1]:
import torch # only for tensor operations, NOT for building the model
import pandas as pd
from sklearn.datasets import load_diabetes 

In [2]:
class DecisionTreeNode:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for splitting
        self.left = left                    # Left child node
        self.right = right                  # Right child node
        self.value = value                  # Value for leaf nodes (class label or regression value)


class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        best_split_error = float('inf')
        best_feature_index = None
        best_threshold = None
        if num_samples < self.min_samples_split or (self.max_depth is not None and depth >= self.max_depth):
            return DecisionTreeNode(value=torch.mean(y))
        for feature_index in range(num_features):
            thresholds = torch.unique(X[:, feature_index])
            for threshold in thresholds:
                left_indices = X[:, feature_index] <= threshold
                right_indices = X[:, feature_index] > threshold
                y_left = torch.mean(y[left_indices])
                y_right = torch.mean(y[right_indices])
                if len(y[left_indices]) == 0 or len(y[right_indices]) == 0:
                    continue
                split_error = 0
                if len(y[left_indices]) > 0:
                    split_error += torch.sum((y[left_indices] - y_left) ** 2)
                if len(y[right_indices]) > 0:
                    split_error += torch.sum((y[right_indices] - y_right) ** 2)
                if split_error <= best_split_error:
                    best_split_error = split_error
                    best_feature_index = feature_index
                    best_threshold = threshold
        left_indices = X[:, best_feature_index] <= best_threshold
        right_indices = X[:, best_feature_index] > best_threshold
        left_subtree = self.build_tree(X[left_indices], y[left_indices], depth + 1)
        right_subtree = self.build_tree(X[right_indices], y[right_indices], depth + 1)
        return DecisionTreeNode(feature_index=best_feature_index, threshold=best_threshold, left=left_subtree, right=right_subtree)

    def fit_tree(self, X, y):
        self.root = self.build_tree(X, y)

    def print_tree(self, node=None, depth=0):
        if node is None:
            node = self.root
        if node.value is not None:
            print(f"{('|' + ' '*4)*depth}Leaf: {node.value}")
        else:
            print(f"{('|' + ' '*4)*depth}Feature {node.feature_index} <= {node.threshold}")
            self.print_tree(node.left, depth + 1)
            print(f"{('|' + ' '*4)*depth}Feature {node.feature_index} > {node.threshold}")
            self.print_tree(node.right, depth + 1)

    def predict(self, X, node):
        if node.value is not None:
            return node.value
        feature_value = X[node.feature_index]
        if feature_value <= node.threshold:
            return self.predict(X, node.left)
        else:
            return self.predict(X, node.right)
        
    def predictions(self, X):
        preds = []
        for i in range(X.shape[0]):
            pred = self.predict(X[i], self.root)
            if isinstance(pred, torch.Tensor):
                pred = pred.item()
            preds.append(pred)
        return torch.tensor(preds, dtype=torch.float32)
    
    def evaluate(self, X, y):
        predictions = []
        for i in range(X.shape[0]):
            pred = self.predict(X[i], self.root)
            if isinstance(pred, torch.Tensor):
                pred = pred.item()
            predictions.append(pred)
        predictions = torch.tensor(predictions, dtype=torch.float32)
        mse = torch.mean((predictions - y) ** 2).item()
        return mse
        
        
    
class RandomForest:
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, num_resampled_features=None, num_resampled_samples=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.num_resampled_features = num_resampled_features
        self.num_resampled_samples = num_resampled_samples
        self.trees = []  # will store tuples (tree, feature_indices)

    def build_forest(self, X, y):
        n_total_features = X.shape[1]
        # default number of features to sample per tree: sqrt(n_features)
        if self.num_resampled_features is None:
            self.num_resampled_features = max(1, int((n_total_features) ** 0.5))
        self.num_resampled_features = min(self.num_resampled_features, n_total_features)

        for _ in range(self.n_estimators):
            n_samples = self.num_resampled_samples or X.shape[0]
            indices = torch.randperm(X.shape[0])[:n_samples]
            X_bootstrap = X[indices]
            y_bootstrap = y[indices]
            feature_indices = torch.randperm(n_total_features)[:self.num_resampled_features]
            tree = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit_tree(X_bootstrap[:, feature_indices], y_bootstrap)
            self.trees.append((tree, feature_indices))

    def predict(self, x):
        if len(self.trees) == 0:
            raise ValueError("Forest has no trees. Call build_forest first.")
        prediction = 0.0
        for tree, feature_indices in self.trees:
            # collect predictions for this tree using the same feature subset used in training
            tree_preds = []
            x_sub = x[feature_indices]
            p = tree.predict(x_sub, tree.root)
            # ensure scalar tensor / float
            if isinstance(p, torch.Tensor):
                p = p.item()
            tree_preds.append(p)
            prediction += torch.tensor(tree_preds, dtype=torch.float32)
        return prediction / len(self.trees)
    
    def predictions(self, X):
        preds = []
        for i in range(X.shape[0]):
            pred = self.predict(X[i])
            if isinstance(pred, torch.Tensor):
                pred = pred.item()
            preds.append(pred)
        return torch.tensor(preds, dtype=torch.float32)
    
    def evaluate(self, X, y):
        predictions = self.predictions(X)
        mse = torch.mean((predictions - y) ** 2).item()
        return mse



In [3]:
from sklearn.model_selection import train_test_split

diabetes = load_diabetes(scaled=False, as_frame=True)
diabetes_df = diabetes.frame

X = diabetes_df.drop("target", axis=1).values
y = diabetes_df["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


In [4]:

tree = DecisionTree(max_depth=3)
tree.fit_tree(X_train, y_train)
tree.print_tree()
tree_predictions = tree.predictions(X_test)
print(f"Tree MSE: {tree.evaluate(X_test, y_test)}")
forest = RandomForest(n_estimators=100, max_depth=3, num_resampled_features=8, num_resampled_samples=8)
forest.build_forest(X_train, y_train)
forest_predictions = forest.predictions(X_test)
print(f"Random Forest MSE: {forest.evaluate(X_test, y_test)}")

Feature 2 <= 26.799999237060547
|    Feature 8 <= 4.700500011444092
|    |    Feature 8 <= 4.158899784088135
|    |    |    Leaf: 80.87754821777344
|    |    Feature 8 > 4.158899784088135
|    |    |    Leaf: 109.92233276367188
|    Feature 8 > 4.700500011444092
|    |    Feature 7 <= 6.0
|    |    |    Leaf: 159.57408142089844
|    |    Feature 7 > 6.0
|    |    |    Leaf: 256.3333435058594
Feature 2 > 26.799999237060547
|    Feature 2 <= 33.099998474121094
|    |    Feature 9 <= 99.0
|    |    |    Leaf: 175.8000030517578
|    |    Feature 9 > 99.0
|    |    |    Leaf: 230.51515197753906
|    Feature 2 > 33.099998474121094
|    |    Feature 5 <= 129.1999969482422
|    |    |    Leaf: 291.22222900390625
|    |    Feature 5 > 129.1999969482422
|    |    |    Leaf: 225.75
Tree MSE: 3656.187255859375
Random Forest MSE: 3317.71337890625
